In [1]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
numsw = 80
graphfile_list = ["ls","dring_80_64.edgelist","rrg_80_64.edgelist","df_p40_a2_h19.edgelist"]
# graphfile_list = ["ls"] -> adjust when computing
lfname_list = ["leafspine","dring","rrg","df2"]
nlinks_list = [1024,1066,1024,1560//2]
failpct_list = range(2,11,2)
failseed_list = range(5) # range(10)

In [ ]:
from collections import deque

def compute_path_metrics(link, source_range=None, dest_range=None):
    num_nodes = len(link)
    total_distance = 0
    reachable_pairs = 0
    diameter = 0  # Longest shortest path

    def bfs(start):
        visited = [False] * num_nodes
        distance = [float('inf')] * num_nodes
        queue = deque()

        visited[start] = True
        distance[start] = 0
        queue.append(start)

        while queue:
            current = queue.popleft()
            for neighbor in range(num_nodes):
                if link[current][neighbor] == 1 and not visited[neighbor]:
                    visited[neighbor] = True
                    distance[neighbor] = distance[current] + 1
                    queue.append(neighbor)
        return distance

    # Define source and destination ranges
    source_nodes = source_range if source_range else range(num_nodes)
    dest_nodes = dest_range if dest_range else range(num_nodes)

    for i in source_nodes:
        dist = bfs(i)
        for j in dest_nodes:
            if i != j and dist[j] != float('inf'):
                total_distance += dist[j]
                reachable_pairs += 1
                diameter = max(diameter, dist[j])

    avg_path_length = total_distance / reachable_pairs if reachable_pairs > 0 else float('inf')
    return avg_path_length, diameter


In [5]:
with open("data.txt", 'w') as fwrite:
    for ig,gfile in enumerate(graphfile_list):
        nlinks = nlinks_list[ig]*2
        lfname = lfname_list[ig]
        for failpct in failpct_list:
            numfaillinks = int(nlinks * failpct / 100)
            for failseed in failseed_list:
                linkfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_link/linkfailurefiles/{lfname}_{numfaillinks}_{failseed}.lf"
        
                if gfile == "ls":
                    link = list()
                    for i in range(numsw):
                        link.append(list())
                        for j in range(numsw):
                            link[i].append(0)
                    for spine in range(0,16):
                        for leaf in range(16,80):
                            link[spine][leaf] = 1
                            link[leaf][spine] = 1

                    with open(linkfailurefile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split()
                            leaf = int(tokens[0])+16
                            spine = int(tokens[1])
                            direction = int(tokens[2])
                            # if link[leaf][spine] != 1:
                            #     print(f"ERROR: {lfname} should have a link from {leaf} to {spine} but not")
                            # else:
                            if direction == 0:
                                if link[leaf][spine] != 1:
                                    print(f"ERROR: {lfname} should have a link from {leaf} to {spine} but not")
                                link[leaf][spine] = 0
                            else:
                                if link[spine][leaf] != 1:
                                    print(f"ERROR: {lfname} should have a link from {spine} to {leaf} but not")
                                link[spine][leaf] = 0

                else:
                    graphfile = f"/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/evaltopologyfiles/{gfile}"
                    # read graphfile
                    link = list()
                    for i in range(numsw):
                        link.append(list())
                        for j in range(numsw):
                            link[i].append(0)
                    with open(graphfile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split("->")
                            fromsw = int(tokens[0])
                            tosw = int(tokens[1])
                            link[fromsw][tosw] = 1
                            link[tosw][fromsw] = 1

                    # read linkfailurefile (if needed)
                    with open(linkfailurefile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split()
                            fromsw = int(tokens[0])
                            tosw = int(tokens[1])
                            if link[fromsw][tosw] != 1:
                                print(f"ERROR: {lfname} should have a link from {fromsw} to {tosw} but not")
                            else:
                                link[fromsw][tosw] = 0

                fwrite.write(f"graph {lfname} failpct {failpct} failseed {failseed} ")
                
                if gfile == "ls":
                    source_range = range(16, 80)
                    dest_range = range(16, 80)
                else:
                    source_range = range(80)
                    dest_range = range(80)

                avgplen, diam = compute_path_metrics(link, source_range, dest_range)
                fwrite.write(f"avgplen {avgplen:.4f} diameter {diam}\n")

diameter with link failure, average shortest path length, number of paths with length <= 3

In [2]:
from collections import deque

def compute_path_metrics(link):
    num_nodes = len(link)
    total_distance = 0
    reachable_pairs = 0
    diameter = 0  # Longest shortest path

    def bfs(start):
        visited = [False] * num_nodes
        distance = [float('inf')] * num_nodes
        queue = deque()

        visited[start] = True
        distance[start] = 0
        queue.append(start)

        while queue:
            current = queue.popleft()
            for neighbor in range(num_nodes):
                if link[current][neighbor] == 1 and not visited[neighbor]:
                    visited[neighbor] = True
                    distance[neighbor] = distance[current] + 1
                    queue.append(neighbor)
        return distance

    for i in range(num_nodes):
        dist = bfs(i)
        for j in range(num_nodes):
            if i != j and dist[j] != float('inf'):
                total_distance += dist[j]
                reachable_pairs += 1
                diameter = max(diameter, dist[j])  # Track longest shortest path

    avg_path_length = total_distance / reachable_pairs if reachable_pairs > 0 else float('inf')
    return avg_path_length, diameter




from collections import defaultdict

def count_simple_paths(link, max_length=4):
    num_nodes = len(link)

    # ✅ Convert adjacency matrix to adjacency list
    adj_list = {i: [j for j in range(num_nodes) if link[i][j] == 1] for i in range(num_nodes)}

    # ✅ Initialize path count storage
    path_counts = {length: [[0]*num_nodes for _ in range(num_nodes)] for length in range(1, max_length+1)}

    # ✅ Memoization cache for visited sets (optional, useful for larger graphs)
    visited_cache = defaultdict(set)

    def dfs(start, current, visited, depth):
        if depth > max_length:
            return

        if depth > 0:
            path_counts[depth][start][current] += 1

        for neighbor in adj_list[current]:
            if neighbor in visited:
                continue  # ✅ Early pruning: skip loops
            dfs(start, neighbor, visited | {neighbor}, depth + 1)

    # ✅ Run DFS from each node
    for node in range(num_nodes):
        dfs(node, node, {node}, 0)

    return path_counts


In [ ]:
with open("data.txt", 'w') as fwrite:
    for ig,gfile in enumerate(graphfile_list):
        nlinks = nlinks_list[ig]*2
        lfname = lfname_list[ig]
        for failpct in failpct_list:
            numfaillinks = int(nlinks * failpct / 100)
            for failseed in failseed_list:
                linkfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_link/linkfailurefiles/{lfname}_{numfaillinks}_{failseed}.lf"
        
                if gfile == "ls":
                    link = list()
                    for i in range(numsw):
                        link.append(list())
                        for j in range(numsw):
                            link[i].append(0)
                    for spine in range(0,16):
                        for leaf in range(16,80):
                            link[spine][leaf] = 1
                            link[leaf][spine] = 1

                    with open(linkfailurefile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split()
                            leaf = int(tokens[0])+16
                            spine = int(tokens[1])
                            direction = int(tokens[2])
                            if link[leaf][spine] != 1:
                                print(f"ERROR: {lfname} should have a link from {leaf} to {spine} but not")
                            else:
                                if direction == 0:
                                    link[leaf][spine] = 0
                                else:
                                    link[spine][leaf] = 0

                else:
                    graphfile = f"/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/evaltopologyfiles/{gfile}"
                    # read graphfile
                    link = list()
                    for i in range(numsw):
                        link.append(list())
                        for j in range(numsw):
                            link[i].append(0)
                    with open(graphfile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split("->")
                            fromsw = int(tokens[0])
                            tosw = int(tokens[1])
                            link[fromsw][tosw] = 1
                            link[tosw][fromsw] = 1

                    # read linkfailurefile (if needed)
                    with open(linkfailurefile,'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            tokens = line.split()
                            fromsw = int(tokens[0])
                            tosw = int(tokens[1])
                            if link[fromsw][tosw] != 1:
                                print(f"ERROR: {lfname} should have a link from {fromsw} to {tosw} but not")
                            else:
                                link[fromsw][tosw] = 0

                fwrite.write(f"graph {lfname} failpct {failpct} failseed {failseed} ")

                avg_length, longest_shortest = compute_path_metrics(link)
                fwrite.write(f"avgplen {avg_length:.4f} diameter {longest_shortest} ")

                path_counts = count_simple_paths(link, max_length=3)
                if gfile == "ls":
                    mynumswstart = 16
                else:
                    mynumswstart = 0
                for i in range(1,4):
                    sumnumpath = 0
                    countnumpath = 0
                    for fromsw in range(mynumswstart,numsw):
                        for tosw in range(mynumswstart,numsw):
                            if fromsw!=tosw and path_counts[i][fromsw][tosw]>0:
                                sumnumpath += path_counts[i][fromsw][tosw]
                                countnumpath += 1
                    fwrite.write(f"pcount{i} {sumnumpath/countnumpath if countnumpath > 0 else 0:.4f} ")

                fwrite.write("\n")

ERROR: leafspine should have a link from 63 to 8 but not
ERROR: leafspine should have a link from 67 to 13 but not
ERROR: leafspine should have a link from 19 to 0 but not
ERROR: leafspine should have a link from 63 to 8 but not
ERROR: leafspine should have a link from 58 to 14 but not
ERROR: leafspine should have a link from 74 to 12 but not
ERROR: leafspine should have a link from 22 to 10 but not
ERROR: leafspine should have a link from 67 to 13 but not
ERROR: leafspine should have a link from 19 to 0 but not
ERROR: leafspine should have a link from 55 to 9 but not
ERROR: leafspine should have a link from 63 to 5 but not
ERROR: leafspine should have a link from 19 to 7 but not
ERROR: leafspine should have a link from 63 to 8 but not
ERROR: leafspine should have a link from 58 to 14 but not
ERROR: leafspine should have a link from 34 to 13 but not
ERROR: leafspine should have a link from 74 to 12 but not
ERROR: leafspine should have a link from 22 to 10 but not
ERROR: leafspine shoul